# Atomic structures: build, select, edit and validate

Worked examples of the `discuspdf.Structure` API: construction, validation,
select/edit, distances, and wrapping.

In [1]:
import numpy as np
from discuspdf import Lattice, Structure

## GaAs conventional cell

Zinc blende, a = 4.53 Å. Ga at (0, 0, 0), As at (0.25, 0.25, 0.25).

In [2]:
a = 4.53
# Lattice.from_parameters — cell from the six parameters, canonical orientation
#   a, b, c              lengths in Å, finite and > 0
#   alpha, beta, gamma   degrees, strictly between 0 and 180; alpha = ∠(b, c), beta = ∠(a, c), gamma = ∠(a, b)
#   orientation          a ∥ x, b in the xy plane, right-handed (use Lattice.from_matrix for any other)
#   raises ValueError    for bad lengths/angles or an impossible (degenerate) cell
#   returns              an immutable Lattice
lat = Lattice.from_parameters(a, a, a, 90.0, 90.0, 90.0)
# Structure parameters (fractional coordinates; distances in Å) — immutable, validated on construction
#   lattice, species, frac   Lattice; (N,) labels (≤ 8 chars, no whitespace); (N, 3) finite fractional coords
#   model           required keyword: "average" (site occupancies) or "configuration" (all occupancies 1.0)
#   occupancy       (N,) values in (0, 1]; default all 1.0; rows sharing a site must sum to ≤ 1
#   site_label      (N,) site names (≤ 16 chars, no whitespace, '|' or ','); rows with the same label and
#                   cell_index form one site (average model: same position); default "Ga1", "As1", ...
#   atom_id         (N,) unique ints ≥ 0, default 0..N-1; used by distance(), select(ids=), remove()
#   b_iso           (N,) isotropic B factors in Å², ≥ 0 (default None)
#   supercell       (3,) repetitions ≥ 1 (default (1, 1, 1)); cell_index (N, 3) cell of each atom (default 0)
#   pbc             3 booleans, periodic axes (default all True)
#   min_separation  Å; different-site atoms closer than this raise ValueError (average model: only if the
#                   two sites' occupancies sum > 1); default 0.1, None skips the check
gaas = Structure(
    lat,
    np.array(["Ga", "As"]),
    np.array([[0.0, 0.0, 0.0], [0.25, 0.25, 0.25]]),
    model="configuration",
)
print(gaas)
# Structure.counts — dict species -> number of rows
print(f"counts: {gaas.counts()}")
# Structure.composition — dict species -> occupancy-weighted count
print(f"composition: {gaas.composition()}")
# Structure.formula — formula string from composition(), e.g. 'GaAs' or 'Cu0.75Au0.25'
print(f"formula: {gaas.formula()}")
# Structure.atomic_numbers — (N,) int array of atomic numbers; an unknown species label raises ValueError here
print(f"atomic numbers: {gaas.atomic_numbers()}")

Structure(n=2, formula=GaAs, model='configuration', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=4.53 Å, b=4.53 Å, c=4.53 Å, alpha=90°, beta=90°, gamma=90°)
counts: {'Ga': 1, 'As': 1}
composition: {'Ga': 1.0, 'As': 1.0}
formula: GaAs
atomic numbers: [31 33]


In [3]:
# Ga-As distance
# Structure.distance — distance in Å between two atoms (minimum image if periodic)
#   id_a, id_b   atom_id values (not row numbers); unknown ids raise ValueError; mixed pbc not supported
d = gaas.distance(0, 1)
print(f"Ga-As distance: {d:.6f} Å  (expected {a * np.sqrt(3) / 4:.6f})")

Ga-As distance: 1.961548 Å  (expected 1.961548)


## Cu/Au average model

FCC, a = 4.0 Å. Mixed site: Cu 0.75, Au 0.25.

In [4]:
a_cu = 4.0
lat_cu = Lattice.from_parameters(a_cu, a_cu, a_cu, 90.0, 90.0, 90.0)
cupau = Structure(
    lat_cu,
    np.array(["Cu", "Au"]),
    np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]),
    model="average",
    occupancy=np.array([0.75, 0.25]),
    site_label=np.array(["S1", "S1"]),
)
print(cupau)
print(f"composition: {cupau.composition()}")
print(f"formula: {cupau.formula()}")

Structure(n=2, formula=Cu0.75Au0.25, model='average', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=4 Å, b=4 Å, c=4 Å, alpha=90°, beta=90°, gamma=90°)
composition: {'Cu': 0.75, 'Au': 0.25}
formula: Cu0.75Au0.25


## Select and edit

Functional edits return new `Structure` instances; the original is untouched.

In [5]:
# Select Ga only
# Structure.select — new Structure with the chosen rows (atom_ids and order kept)
#   give one selector; if several are given, the first of mask, ids, rows, species, site is used
#   mask         (N,) booleans (positional)
#   ids, rows    atom_id values, or row positions
#   species      a label or list of labels, e.g. "Ga" or ["Cu", "Au"]
#   site         a site_label
#   allow_empty  default False: an empty selection raises ValueError
ga_only = gaas.select(species="Ga")
print(f"Select Ga only: {ga_only}")

# Edit: move As
# Structure.with_frac — new Structure with new fractional coordinates, fully re-validated
#   frac            (N, 3) fractional coordinates
#   min_separation  overlap check distance in Å (default 0.1; None skips)
gaas2 = gaas.with_frac(np.array([[0.0, 0.0, 0.0], [0.3, 0.25, 0.25]]))
d2 = gaas2.distance(0, 1)
print(f"After moving As: Ga-As distance = {d2:.6f} Å")

# Original unchanged
d_orig = gaas.distance(0, 1)
print(f"Original Ga-As distance: {d_orig:.6f} Å (unchanged)")

Select Ga only: Structure(n=1, formula=Ga, model='configuration', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=4.53 Å, b=4.53 Å, c=4.53 Å, alpha=90°, beta=90°, gamma=90°)
After moving As: Ga-As distance = 2.100475 Å
Original Ga-As distance: 1.961548 Å (unchanged)


## Wrap

`wrapped()` returns positions in [0, 1) for fully periodic structures.

In [6]:
# Structure.wrapped — new Structure with frac wrapped to [0, 1); cell_index unchanged
#   only for fully periodic structures (otherwise ValueError)
gaas_wrapped = gaas.wrapped()
# Structure.frac — (N, 3) fractional coordinates (read-only view)
print(f"Wrapped: {gaas_wrapped.frac}")

Wrapped: [[0.   0.   0.  ]
 [0.25 0.25 0.25]]


## Validation

Common validation errors:

In [7]:
# Configuration with occupancy < 1 raises
try:
    Structure(lat, np.array(["Ga"]), np.array([[0.25, 0.25, 0.25]]),
              model="configuration", occupancy=np.array([0.5]))
except ValueError as e:
    print(f"Configuration occ < 1: {e}")

# Site occupancy sum > 1 raises
try:
    Structure(lat, np.array(["Cu", "Au"]),
              np.array([[0.25, 0.25, 0.25], [0.25, 0.25, 0.25]]),
              model="average", occupancy=np.array([0.7, 0.5]),
              site_label=np.array(["S1", "S1"]))
except ValueError as e:
    print(f"Site occ sum > 1: {e}")

Configuration occ < 1: configuration model: all occupancies must be exactly 1.0; for partial occupancies use model='average'
Site occ sum > 1: average model: site 'S1' in cell (0, 0, 0) has total occupancy 1.200000 > 1 + 1e-6
